In [ ]:
from tests.adapters import run_get_response_log_probs, run_tokenize_prompt_and_output, run_sft_microbatch_train_step, get_packed_sft_dataset, run_iterate_batches
from torch import Tensor
from tqdm import tqdm
import torch
import numpy as np
import random
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
!gzip -dk train.jsonl.gz

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
device = torch.device("cuda")

num_epochs = 1
batch_size = 32
gradient_accumulation_steps = 4
seq_length = 512
sft_sample_path = "train.jsonl"

In [ ]:
!pip install --upgrade huggingface_hub
from huggingface_hub import login
login()

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    'meta-llama/Llama-3.1-8B',
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
).to(device)
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.1-8B')

In [ ]:
packed_sft_dataset = get_packed_sft_dataset(
    tokenizer=tokenizer,
    dataset_path=sft_sample_path,
    seq_length=seq_length,
    shuffle=True,
)
train_loader = run_iterate_batches(
    dataset=packed_sft_dataset, batch_size=batch_size, shuffle=True
)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.1)


In [ ]:
global_step = -1

for epoch in range(num_epochs):
    model.train()

    for idx, batch in tqdm(enumerate(train_loader)):
        input_ids = batch["input_ids"]
        labels = batch["labels"]
        mask = torch.full_like(input_ids, True) # mask is not necessary. Creating an all True mask to reuse existing code
        log_probs = run_get_response_log_probs(model, input_ids, labels, False)['log_probs']
        loss, _ = run_sft_microbatch_train_step(log_probs, mask, gradient_accumulation_steps)
        global_step += 1
        
        if (idx + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            
        print(f"Step {global_step:06d}: Train loss: {loss.cpu().item()}")


In [ ]:
import os

output_dir = "sft_safety_model"
os.makedirs(output_dir, exist_ok=True)

print("saving the model and tokenizer...")
model.save_pretrained(save_directory=output_dir)
tokenizer.save_pretrained(save_directory=output_dir)